# 遗传算法求解华容道问题：学生练习版


本练习基于完成版 `huarongdao_genetic_algorithm.ipynb` 制作，用于考查学生对遗传算法求解华容道问题的掌握情况。

本实验重点展示：

1. 如何把华容道移动序列编码成染色体。
2. 如何根据动作序列模拟棋盘变化。
3. 如何设计适应度函数评价一个候选解。
4. 如何实现选择、交叉、变异等遗传算法操作。
5. 如何输出遗传算法找到的完整操作路径。

说明：经典华容道搜索空间很大，完全随机的遗传算法很难稳定找到完整解。为了让教学实验可复现，本程序在初始种群中加入一个可行种子个体，然后通过遗传算法框架进行评价和输出。这样可以稳定展示遗传算法求解流程，同时避免课堂运行时间不可控。


## 1. 遗传算法求解华容道的建模方法

遗传算法需要先把问题转换成“染色体”的形式。

在本实验中：

- 一个染色体表示一串移动动作。
- 每个基因表示一步操作：`(棋子编号, 移动方向)`。
- 例如 `(0, 'Down')` 表示让 0 号棋子曹操向下移动一步。

对一个染色体进行评价时，程序会：

1. 从初始棋盘开始。
2. 依次尝试执行染色体中的动作。
3. 非法动作不执行，并给予一定惩罚。
4. 如果某一步使曹操到达出口，则该染色体被认为找到解。
5. 如果没有到达出口，则根据曹操距离目标的远近、阻挡数量、合法动作数量等计算适应度。

遗传算法的核心操作：

- **选择**：优先保留适应度高的个体。
- **交叉**：把两个个体的动作序列片段组合成新个体。
- **变异**：随机修改个体中的某些动作。


## 2. 遗传算法求解华容道问题的实现步骤

实现本程序时，可以按照下面流程理解遗传算法如何求解华容道问题。练习版中需要学生补全的函数会在代码中用 `TODO` 明确标出。

1. **定义棋盘和棋子**：设置棋盘大小、棋子尺寸、初始状态和曹操出口位置。
2. **设计染色体编码**：把一个候选解表示为一串动作，每个基因对应一次棋子移动。
3. **生成初始种群**：随机生成多条动作序列，并加入一个可行种子个体，保证课堂实验可复现。
4. **解码染色体**：从初始状态开始，依次执行染色体中的动作，记录合法动作和非法动作。
5. **计算适应度**：根据是否到达目标、曹操到出口距离、通道阻挡数量、合法动作数量和非法动作数量评价个体好坏。
6. **选择优秀个体**：使用锦标赛选择，让适应度较高的个体更容易成为父代。
7. **交叉产生后代**：对两个父代动作序列进行单点交叉，组合出新的动作序列。
8. **随机变异**：以一定概率修改后代中的某些动作，增加搜索多样性。
9. **保留精英个体**：每一代保留适应度最高的若干个体，避免优秀结果丢失。
10. **迭代搜索并输出路径**：重复评价、选择、交叉、变异过程，直到找到目标或达到最大迭代次数。

需要注意：遗传算法是一种启发式优化方法，不保证一定找到最短路径。本实验更适合用于观察编码、适应度、选择、交叉和变异如何协同工作。

本练习中需要学生补全的遗传算法相关函数包括：`decode_chromosome`、`evaluate_chromosome`、`random_action`、`random_chromosome`、`tournament_selection`、`crossover`、`mutate` 和 `genetic_algorithm`。


## 3. 遗传算法学生练习程序

下面代码单元是学生练习程序。棋盘建模、移动合法性判断、状态绘制等非遗传算法基础函数已经保留完整实现；遗传算法相关重点函数已经留空，需要学生根据注释和 TODO 提示补全。


In [ ]:
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# ============================================================
# 1. 棋盘与棋子定义
# ============================================================
BOARD_WIDTH = 4
BOARD_HEIGHT = 5
GOAL_CAOCAO_POSITION = (3, 1)
DIRECTIONS = ('Up', 'Down', 'Left', 'Right')

# 棋子定义格式：棋子名称、宽度、高度、显示字符、颜色。
PIECES = (
    ('CaoCao', 2, 2, 'C', '#d94f45'),
    ('ZhangFei', 1, 2, 'Z', '#7b4ab2'),
    ('ZhaoYun', 1, 2, 'Y', '#3b82c4'),
    ('MaChao', 1, 2, 'M', '#2f9e44'),
    ('HuangZhong', 1, 2, 'H', '#8f6b32'),
    ('GuanYu', 2, 1, 'G', '#e0a526'),
    ('Soldier1', 1, 1, 'S1', '#8d99ae'),
    ('Soldier2', 1, 1, 'S2', '#8d99ae'),
    ('Soldier3', 1, 1, 'S3', '#8d99ae'),
    ('Soldier4', 1, 1, 'S4', '#8d99ae'),
)

INITIAL_STATE = (
    (0, 1),  # CaoCao
    (0, 0),  # ZhangFei
    (0, 3),  # ZhaoYun
    (2, 0),  # MaChao
    (2, 3),  # HuangZhong
    (2, 1),  # GuanYu
    (3, 1),  # Soldier1
    (3, 2),  # Soldier2
    (4, 0),  # Soldier3
    (4, 3),  # Soldier4
)


# 函数功能：对华容道状态进行归一化处理。
# 求解作用：把 4 个形状相同的士兵位置排序，减少等价状态带来的重复评价。
# 输入参数：state 表示当前华容道状态，是由所有棋子左上角坐标组成的 tuple。
def normalize_state(state):
    """
    对状态进行归一化。

    输出中仍使用 Soldier1-4 作为士兵名称；但搜索和遗传算法评价时，
    四个士兵形状完全相同，因此将后 4 个士兵位置排序，减少等价差异。
    """
    fixed_pieces = list(state[:6])
    soldiers = sorted(state[6:])
    return tuple(fixed_pieces + soldiers)


INITIAL_STATE = normalize_state(INITIAL_STATE)


# ============================================================
# 2. 基础棋盘函数
# ============================================================
# 函数功能：根据当前状态构造棋盘占用表。
# 求解作用：为移动合法性检查提供每个格子的占用信息。
# 输入参数：state 表示当前华容道状态，是由所有棋子左上角坐标组成的 tuple。
def build_board(state):
    """根据状态构造棋盘占用表。"""
    board = [[None for _ in range(BOARD_WIDTH)] for _ in range(BOARD_HEIGHT)]

    for piece_index, (row, col) in enumerate(state):
        name, width, height, symbol, color = PIECES[piece_index]
        for dr in range(height):
            for dc in range(width):
                r = row + dr
                c = col + dc
                if r < 0 or r >= BOARD_HEIGHT or c < 0 or c >= BOARD_WIDTH:
                    return None
                if board[r][c] is not None:
                    return None
                board[r][c] = piece_index

    return board


# 函数功能：判断曹操是否已经到达出口位置。
# 求解作用：解码染色体时用它判断候选动作序列是否已经成功。
# 输入参数：state 表示需要判断的当前华容道状态。
def is_goal(state):
    """判断曹操是否到达出口。"""
    return state[0] == GOAL_CAOCAO_POSITION


# 函数功能：判断某个棋子能否放置到指定的新位置。
# 求解作用：过滤越界或与其他棋子重叠的非法动作。
# 输入参数：board 是当前棋盘占用表；piece_index 是棋子编号；new_row 和 new_col 是棋子移动后的左上角坐标。
def can_place_piece(board, piece_index, new_row, new_col):
    """判断某个棋子能否移动到新位置。"""
    name, width, height, symbol, color = PIECES[piece_index]

    if new_row < 0 or new_col < 0:
        return False
    if new_row + height > BOARD_HEIGHT or new_col + width > BOARD_WIDTH:
        return False

    for dr in range(height):
        for dc in range(width):
            occupied_by = board[new_row + dr][new_col + dc]
            if occupied_by is not None and occupied_by != piece_index:
                return False

    return True


# 函数功能：尝试在当前状态下执行一个动作基因。
# 求解作用：把染色体中的基因转化为真实棋盘移动，并返回移动是否合法。
# 输入参数：state 是执行动作前的状态；action 是一个动作基因，格式为 (piece_selector, direction)。
def apply_action(state, action):
    """
    尝试对状态执行一个动作。

    参数：
        state: 当前状态。
        action: 一个二元组 (piece_selector, direction)。
                piece_selector 可以是具体棋子编号，也可以是字符串 'Soldier'。

    返回：
        next_state: 执行动作后的状态。如果动作非法，则状态不变。
        is_legal: 该动作是否合法。

    说明：
        当 piece_selector 为 'Soldier' 时，表示“当前任意一个可以按该方向移动的士兵”。
        这适合士兵等价的华容道建模，也让遗传算法种子更稳定。
    """
    piece_selector, direction = action

    if piece_selector == 'Soldier':
        for soldier_index in range(6, 10):
            next_state, is_legal = apply_action(state, (soldier_index, direction))
            if is_legal:
                return next_state, True
        return state, False

    piece_index = piece_selector
    direction_map = {
        'Up': (-1, 0),
        'Down': (1, 0),
        'Left': (0, -1),
        'Right': (0, 1),
    }

    row, col = state[piece_index]
    dr, dc = direction_map[direction]
    new_row = row + dr
    new_col = col + dc

    board = build_board(state)
    if not can_place_piece(board, piece_index, new_row, new_col):
        return state, False

    next_state = list(state)
    next_state[piece_index] = (new_row, new_col)
    return normalize_state(tuple(next_state)), True


# 函数功能：统计曹操下方出口通道中的阻挡格数量。
# 求解作用：作为适应度的一部分，阻挡越少说明局面越接近成功。
# 输入参数：state 表示需要统计出口通道阻挡数量的当前状态。
def obstacle_count(state):
    """统计曹操下方出口通道中的阻挡数量。"""
    caocao_row, caocao_col = state[0]
    board = build_board(state)
    count = 0

    for row in range(caocao_row + 2, BOARD_HEIGHT):
        for col in (1, 2):
            occupied_by = board[row][col]
            if occupied_by is not None and occupied_by != 0:
                count += 1

    return count


# 函数功能：计算曹操当前位置到目标出口位置的曼哈顿距离。
# 求解作用：作为适应度的一部分，距离越小通常说明候选解越好。
# 输入参数：state 表示需要计算曹操到目标距离的当前状态。
def distance_to_goal(state):
    """计算曹操左上角到目标位置的曼哈顿距离。"""
    row, col = state[0]
    goal_row, goal_col = GOAL_CAOCAO_POSITION
    return abs(row - goal_row) + abs(col - goal_col)


# ============================================================
# 3. 染色体、解码与适应度
# ============================================================
# 练习说明：下面这一组函数属于遗传算法核心功能，已经留空作为学生考查内容。
# 学生需要根据每个函数的输入、输出和 TODO 提示完成实现。
# 数据结构功能：保存一个染色体的评价结果。
# 求解作用：集中记录适应度、是否到达目标、最终状态和动作统计信息。
# 字段说明：fitness 为适应度；reached_goal 表示是否到达目标；final_state 为最终状态；valid_actions 为合法动作序列；valid_count 和 illegal_count 统计合法/非法动作数量；first_goal_step 记录首次到达目标的基因步数。
@dataclass
class EvaluationResult:
    """保存一个染色体的评价结果。"""
    fitness: float
    reached_goal: bool
    final_state: tuple
    valid_actions: list
    valid_count: int
    illegal_count: int
    first_goal_step: int | None


# 函数功能：从初始状态开始依次执行染色体中的动作序列。
# 求解作用：得到该染色体最终局面、合法动作列表、非法动作数量和首次到达目标的步数。
# 输入参数：chromosome 表示一个候选解，是由多个动作基因组成的列表。
def decode_chromosome(chromosome):
    """
    解码染色体。需要学生补全。

    输入：
        chromosome: 一个候选解，由多个动作基因组成。

    返回：
        final_state: 执行动作序列后的最终状态。
        valid_actions: 实际成功执行的合法动作列表。
        illegal_count: 非法动作数量。
        first_goal_step: 第一次到达目标时对应的基因步数；如果没有到达目标，则为 None。

    实现提示：
        1. 从 INITIAL_STATE 开始。
        2. 依次取出 chromosome 中的每个 action。
        3. 调用 apply_action(state, action) 执行动作。
        4. 合法动作需要更新 state，并记录到 valid_actions。
        5. 非法动作不更新 state，只增加 illegal_count。
        6. 每一步后调用 is_goal(state) 判断是否到达目标。
    """
    # TODO 1: 初始化 state、valid_actions 和 illegal_count。

    # TODO 2: 遍历染色体中的动作，逐步执行并统计合法/非法动作。

    # TODO 3: 如果到达目标，返回当前状态、合法动作、非法动作数和首次到达目标步数。

    # TODO 4: 如果所有动作执行完仍未到达目标，返回最终状态和 None。
    pass

# 函数功能：计算一个染色体的适应度评分。
# 求解作用：为选择操作提供评价依据，适应度越高的动作序列越可能被保留。
# 输入参数：chromosome 表示需要评价的候选动作序列。
def evaluate_chromosome(chromosome):
    """
    计算染色体适应度。需要学生补全。

    输入：
        chromosome: 需要评价的候选动作序列。

    返回：
        EvaluationResult 对象，保存适应度、是否到达目标、最终状态和动作统计信息。

    实现提示：
        1. 先调用 decode_chromosome(chromosome) 得到执行结果。
        2. 如果到达目标，应给较高适应度，并偏好更短路径。
        3. 如果未到达目标，可以综合 distance_to_goal、obstacle_count、合法动作数和非法动作数进行评分。
        4. 最后返回 EvaluationResult。
    """
    # TODO 5: 调用 decode_chromosome 得到 final_state、valid_actions、illegal_count、first_goal_step。

    # TODO 6: 如果已经到达目标，计算成功奖励 fitness，并返回 EvaluationResult。

    # TODO 7: 如果没有到达目标，根据距离、阻挡数量、合法动作数和非法动作数计算 fitness。

    # TODO 8: 返回 EvaluationResult。
    pass

# 函数功能：随机生成一个动作基因。
# 求解作用：用于初始化随机染色体，也用于变异时替换某个动作。
# 输入参数：无。该函数直接从棋子集合和方向集合中随机采样。
def random_action():
    """
    随机生成一个动作基因。需要学生补全。

    输入：无。
    返回：action，一个二元组 (piece_index, direction)。

    实现提示：随机选择棋子编号和移动方向。
    """
    # TODO 9: 随机选择棋子编号和移动方向，返回动作基因。
    pass

# 函数功能：随机生成指定长度的染色体。
# 求解作用：用于构造遗传算法的初始种群。
# 输入参数：length 表示染色体长度，也就是动作基因的数量。
def random_chromosome(length):
    """
    随机生成一个染色体。需要学生补全。

    输入：
        length: 染色体长度，也就是动作基因数量。

    返回：
        chromosome: 由 length 个随机动作组成的列表。
    """
    # TODO 10: 生成包含 length 个随机动作的列表。
    pass

# ============================================================
# 4. 遗传算法操作：选择、交叉、变异
# ============================================================
# 函数功能：使用锦标赛选择从种群中选出一个较优个体。
# 求解作用：让高适应度个体更容易参与繁殖，同时保留一定随机性。
# 输入参数：population 是当前种群；evaluations 是对应适应度结果；tournament_size 是每次随机竞争的个体数量。
def tournament_selection(population, evaluations, tournament_size=4):
    """
    锦标赛选择。需要学生补全。

    输入：
        population: 当前种群，是多个染色体组成的列表。
        evaluations: 与 population 一一对应的评价结果。
        tournament_size: 每次随机抽取参与竞争的个体数量。

    返回：
        selected_chromosome: 抽样个体中适应度最高的染色体。
    """
    # TODO 11: 随机抽取候选个体下标。

    # TODO 12: 根据适应度选择最优候选个体。

    # TODO 13: 返回被选中的染色体。
    pass

# 函数功能：对两个父代染色体执行单点交叉。
# 求解作用：组合两个候选解的动作片段，产生新的后代个体。
# 输入参数：parent_a 和 parent_b 分别表示两个父代染色体，二者长度需要一致。
def crossover(parent_a, parent_b):
    """
    单点交叉。需要学生补全。

    输入：
        parent_a: 第一个父代染色体。
        parent_b: 第二个父代染色体。

    返回：
        child: 由两个父代片段组合得到的子代染色体。
    """
    # TODO 14: 检查两个父代长度是否一致。

    # TODO 15: 如果染色体长度太短，直接返回 parent_a 的拷贝。

    # TODO 16: 随机选择交叉点，并拼接生成 child。
    pass

# 函数功能：按照给定概率随机修改染色体中的动作。
# 求解作用：增加种群多样性，帮助算法跳出局部最优。
# 输入参数：chromosome 是待变异染色体；mutation_rate 是每个基因发生变异的概率。
def mutate(chromosome, mutation_rate=0.04):
    """
    随机变异。需要学生补全。

    输入：
        chromosome: 待变异染色体。
        mutation_rate: 每个基因发生变异的概率。

    返回：
        child: 变异后的新染色体。
    """
    # TODO 17: 复制染色体。

    # TODO 18: 遍历每个基因，并按 mutation_rate 决定是否替换为随机动作。

    # TODO 19: 返回变异后的染色体。
    pass

# ============================================================
# 5. 可复现实验种子
# ============================================================
# 经典布局纯随机 GA 很难稳定找到完整解。
# 这里提供一个可行种子个体，保证课堂实验可以稳定展示 GA 的完整流程。
SEED_ACTION_NAMES = [
    ('Soldier3', 'Right'), ('MaChao', 'Down'), ('GuanYu', 'Left'), ('Soldier2', 'Up'),
    ('Soldier4', 'Left'), ('HuangZhong', 'Down'), ('Soldier4', 'Right'), ('GuanYu', 'Right'),
    ('MaChao', 'Up'), ('Soldier1', 'Left'), ('Soldier3', 'Left'), ('HuangZhong', 'Left'),
    ('Soldier4', 'Down'), ('GuanYu', 'Right'), ('Soldier1', 'Up'), ('Soldier2', 'Up'),
    ('Soldier4', 'Right'), ('MaChao', 'Down'), ('Soldier1', 'Left'), ('Soldier2', 'Down'),
    ('GuanYu', 'Left'), ('ZhaoYun', 'Down'), ('ZhaoYun', 'Down'), ('CaoCao', 'Right'),
    ('ZhangFei', 'Right'), ('Soldier3', 'Up'), ('MaChao', 'Up'), ('Soldier4', 'Left'),
    ('Soldier3', 'Up'), ('MaChao', 'Up'), ('Soldier4', 'Left'), ('HuangZhong', 'Left'),
    ('Soldier1', 'Left'), ('ZhaoYun', 'Down'), ('GuanYu', 'Right'), ('HuangZhong', 'Up'),
    ('Soldier2', 'Up'), ('Soldier1', 'Right'), ('Soldier3', 'Right'), ('HuangZhong', 'Down'),
    ('ZhangFei', 'Down'), ('Soldier2', 'Right'), ('MaChao', 'Up'), ('Soldier4', 'Up'),
    ('HuangZhong', 'Left'), ('ZhangFei', 'Down'), ('ZhangFei', 'Down'), ('Soldier1', 'Right'),
    ('Soldier2', 'Up'), ('GuanYu', 'Left'), ('GuanYu', 'Left'), ('ZhaoYun', 'Up'),
    ('Soldier4', 'Right'), ('Soldier3', 'Down'), ('ZhaoYun', 'Left'), ('Soldier2', 'Up'),
    ('Soldier1', 'Right'), ('ZhaoYun', 'Down'), ('GuanYu', 'Right'), ('GuanYu', 'Right'),
    ('ZhangFei', 'Up'), ('MaChao', 'Down'), ('Soldier4', 'Left'), ('Soldier2', 'Up'),
    ('ZhangFei', 'Up'), ('ZhaoYun', 'Left'), ('Soldier1', 'Left'), ('Soldier3', 'Down'),
    ('GuanYu', 'Down'), ('CaoCao', 'Down'), ('Soldier1', 'Right'), ('Soldier2', 'Right'),
    ('Soldier3', 'Right'), ('Soldier4', 'Right'), ('ZhangFei', 'Up'), ('ZhaoYun', 'Up'),
    ('MaChao', 'Up'), ('HuangZhong', 'Up'), ('Soldier1', 'Left'), ('Soldier2', 'Left'),
    ('Soldier3', 'Left'), ('Soldier4', 'Left'), ('GuanYu', 'Down'), ('CaoCao', 'Down'),
    ('Soldier1', 'Down'), ('Soldier2', 'Right'), ('ZhangFei', 'Right'), ('ZhaoYun', 'Up'),
    ('ZhaoYun', 'Up'), ('CaoCao', 'Left'), ('Soldier3', 'Down'), ('Soldier4', 'Down'),
    ('Soldier1', 'Down'), ('Soldier2', 'Down'), ('ZhangFei', 'Right'), ('ZhaoYun', 'Right'),
    ('MaChao', 'Right'), ('HuangZhong', 'Up'), ('HuangZhong', 'Up'), ('CaoCao', 'Left'),
    ('Soldier3', 'Left'), ('Soldier4', 'Up'), ('GuanYu', 'Up'), ('Soldier1', 'Right'),
    ('Soldier2', 'Right'), ('Soldier3', 'Right'), ('Soldier4', 'Right'), ('CaoCao', 'Down'),
    ('Soldier1', 'Left'), ('Soldier2', 'Left'), ('Soldier3', 'Left'), ('Soldier4', 'Left'),
    ('GuanYu', 'Up'), ('Soldier4', 'Up'), ('Soldier3', 'Right'), ('CaoCao', 'Right'),
]

PIECE_NAME_TO_INDEX = {piece[0]: index for index, piece in enumerate(PIECES)}
SEED_CHROMOSOME = [
    ('Soldier', direction) if piece_name.startswith('Soldier') else (PIECE_NAME_TO_INDEX[piece_name], direction)
    for piece_name, direction in SEED_ACTION_NAMES
]


# ============================================================
# 6. 遗传算法主程序
# ============================================================
# 函数功能：运行完整遗传算法流程求解华容道。
# 求解作用：循环执行评价、选择、交叉、变异和精英保留，寻找能够让曹操到达出口的动作序列。
# 输入参数：population_size 是种群规模；chromosome_length 是染色体长度；
# generations 是最大迭代代数；elite_size 是每代保留的精英数量；mutation_rate 是变异概率；random_seed 用于固定随机结果。
def genetic_algorithm(
    population_size=80,
    chromosome_length=140,
    generations=80,
    elite_size=4,
    mutation_rate=0.04,
    random_seed=42,
):
    """
    使用遗传算法求解华容道。需要学生补全。

    输入：
        population_size: 种群规模。
        chromosome_length: 每个染色体包含的动作数量。
        generations: 最大迭代代数。
        elite_size: 每代直接保留的精英个体数量。
        mutation_rate: 变异概率。
        random_seed: 随机种子，用于保证实验可复现。

    返回：
        best_chromosome: 搜索过程中找到的最佳染色体。
        best_evaluation: 最佳染色体对应的评价结果。
        solved_generation: 找到目标时的代数；如果未找到，则返回最大代数。

    实现提示：
        1. 设置随机种子。
        2. 初始化随机种群。
        3. 将 SEED_CHROMOSOME 加入种群，保证课堂实验稳定。
        4. 每一代先评价所有染色体。
        5. 按适应度排序，记录当前最优个体。
        6. 如果找到目标，立即返回。
        7. 保留 elite_size 个精英个体。
        8. 使用选择、交叉、变异生成新一代。
        9. 达到最大代数后返回历史最佳结果。
    """
    # TODO 20: 设置随机种子，并初始化随机种群。

    # TODO 21: 加入 SEED_CHROMOSOME，长度不足时补随机动作，长度超出时截断。

    # TODO 22: 初始化 best_chromosome 和 best_evaluation。

    # TODO 23: 编写遗传算法主循环，完成评价、排序、更新最优个体。

    # TODO 24: 如果当前最优个体已经到达目标，返回结果。

    # TODO 25: 保留精英个体，并通过选择、交叉、变异生成新种群。

    # TODO 26: 如果达到最大迭代代数仍未成功，返回历史最佳结果。
    pass

# ============================================================
# 6. 运行遗传算法并打印结果
# ============================================================
# 注意：学生完成所有 TODO 后，再将 RUN_EXERCISE 设置为 True 运行。
RUN_EXERCISE = False

if RUN_EXERCISE:
    best_chromosome, best_evaluation, solved_generation = genetic_algorithm()

    print()
    print('遗传算法运行结束。')
    print('找到目标：', best_evaluation.reached_goal)
    print('求解代数：', solved_generation)
    print('合法动作数量：', best_evaluation.valid_count)
    print('非法动作数量：', best_evaluation.illegal_count)

    if best_evaluation.reached_goal:
        print('到达目标所需基因步数：', best_evaluation.first_goal_step)
        print()
        print('遗传算法得到的完整操作路径如下：')

        state = INITIAL_STATE
        print('000. 初始状态')
        printed_step = 0

        for gene_index, action in enumerate(best_chromosome, start=1):
            next_state, is_legal = apply_action(state, action)
            if not is_legal:
                continue

            state = next_state
            printed_step += 1
            piece_selector, direction = action
            piece_name = 'Soldier' if piece_selector == 'Soldier' else PIECES[piece_selector][0]
            print(f'{printed_step:03d}. {piece_name} {direction}')

            if is_goal(state):
                break
    else:
        print('本次遗传算法没有找到完整解，可以增大 generations、population_size 或调整适应度函数。')


# ============================================================
# 7. 可选：绘制结果状态
# ============================================================
# 函数功能：将某个华容道状态绘制成颜色块棋盘。
# 求解作用：用于观察最终状态或中间状态，不参与遗传算法搜索计算。
# 输入参数：state 是需要绘制的华容道状态；title 是图像标题。
def draw_state(state, title='Huarong Dao State'):
    """使用颜色块绘制华容道状态。"""
    fig, ax = plt.subplots(figsize=(4.8, 6.0))
    ax.set_xlim(0, BOARD_WIDTH)
    ax.set_ylim(0, BOARD_HEIGHT)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_facecolor('#f7f2e8')
    ax.set_title(title)

    for x in range(BOARD_WIDTH + 1):
        ax.plot([x, x], [0, BOARD_HEIGHT], color='#5c4a36', linewidth=1)
    for y in range(BOARD_HEIGHT + 1):
        ax.plot([0, BOARD_WIDTH], [y, y], color='#5c4a36', linewidth=1)

    ax.plot([1, 3], [5, 5], color='#b00020', linewidth=5, solid_capstyle='round')
    ax.text(2, 4.82, 'EXIT', ha='center', va='center', color='#b00020', fontsize=10, weight='bold')

    for piece_index, (row, col) in enumerate(state):
        name, width, height, symbol, color = PIECES[piece_index]
        rect = Rectangle((col, row), width, height, facecolor=color, edgecolor='black', linewidth=1.8, alpha=0.92)
        ax.add_patch(rect)
        ax.text(col + width / 2, row + height / 2, symbol, ha='center', va='center', fontsize=12, color='white', weight='bold')

    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.show()


# 完成 TODO 并将 RUN_EXERCISE 设置为 True 后，可以取消下面代码注释查看图像。
# draw_state(INITIAL_STATE, 'Initial State')
# if RUN_EXERCISE and best_evaluation.reached_goal:
#     draw_state(best_evaluation.final_state, 'Final State Found by Genetic Algorithm')


## 4. 实验说明

本实验虽然使用遗传算法框架完成求解，但需要注意：

1. 华容道是强约束、长路径的状态空间问题。
2. 完全随机初始化的遗传算法很难稳定找到 100 多步的精确解。
3. 本程序加入可行种子个体，是为了让课堂实验能够稳定观察遗传算法流程。
4. 如果去掉种子个体，可以尝试增大种群规模、染色体长度、迭代代数，并重新设计适应度函数。

思考题：

1. 遗传算法为什么不一定能找到最短路径？
2. 当前染色体编码方式有什么优点和缺点？
3. 如果非法动作太多，适应度函数应该如何惩罚？
4. 如何设计更好的交叉和变异策略？
5. 对比 A* 算法，遗传算法更适合解决哪类问题？
